# Silver Layer: EV Charging Stations Transformation

This notebook transforms raw EV charging station data from Bronze to Silver layer.

## Transformations Applied:
- **Column Standardization**: All columns converted to snake_case
- **Charger Rating Parsing**: Extract numeric kW values from strings like "50 kW"
- **Speed Categorization**: Classify chargers as Slow (<7kW), Fast (7-22kW), Rapid (22-150kW), Ultra-Rapid (>150kW)
- **Data Quality Flags**: Identify missing critical fields (location, rating)
- **Deduplication**: Remove duplicate records based on station name + location

## Data Quality Validations:
- Count of stations with missing coordinates
- Count of stations with invalid/missing charger ratings
- Duplicate station detection
- Total record counts

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Table paths
BRONZE_TABLE = "mobility_ai.bronze.ev_charging_stations_raw"
SILVER_TABLE = "mobility_ai.silver.ev_charging_stations"

print(f"Source: {BRONZE_TABLE}")
print(f"Target: {SILVER_TABLE}")

In [0]:
# Load bronze data
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze record count: {bronze_df.count()}")

# Standardize column names to snake_case
def to_snake_case(col_name):
    import re
    # Replace spaces and special chars with underscore
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return re.sub(r'[\s-]+', '_', s2).lower()

# Rename all columns
for col in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(col, to_snake_case(col))

# Parse charger rating (extract numeric kW from strings like "50 kW")
silver_df = bronze_df.withColumn(
    "charger_rating_kw",
    F.expr("try_cast(regexp_extract(charger_rating, r'(\\d+\\.?\\d*)', 1) as double)")
)

# Categorize charging speed
silver_df = silver_df.withColumn(
    "charging_speed",
    F.when(F.col("charger_rating_kw") < 7, "Slow")
     .when((F.col("charger_rating_kw") >= 7) & (F.col("charger_rating_kw") < 22), "Fast")
     .when((F.col("charger_rating_kw") >= 22) & (F.col("charger_rating_kw") <= 150), "Rapid")
     .when(F.col("charger_rating_kw") > 150, "Ultra-Rapid")
     .otherwise("Unknown")
)

# Add data quality flags
silver_df = silver_df.withColumn(
    "has_valid_location",
    F.when(
        (F.col("latitude").isNotNull()) & 
        (F.col("longitude").isNotNull()) &
        (F.col("latitude").between(-90, 90)) &
        (F.col("longitude").between(-180, 180)),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_rating",
    F.when(F.col("charger_rating_kw").isNotNull() & (F.col("charger_rating_kw") > 0), True)
     .otherwise(False)
)

# Deduplicate based on station_name + location
window_spec = Window.partitionBy("station_name", "latitude", "longitude").orderBy(F.col("ingestion_timestamp").desc())

silver_df = silver_df.withColumn("row_num", F.row_number().over(window_spec))
silver_df = silver_df.filter(F.col("row_num") == 1).drop("row_num")

# Add processing metadata
silver_df = silver_df.withColumn("silver_processed_at", F.current_timestamp())

print(f"Silver record count after transformations: {silver_df.count()}")
display(silver_df.limit(10))

In [0]:
# Write to silver table with merge/upsert logic
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Successfully wrote {silver_df.count()} records to {SILVER_TABLE}")

In [0]:
# Validate silver data
validation_df = spark.table(SILVER_TABLE)

# Quality metrics
total_count = validation_df.count()
missing_coords = validation_df.filter(F.col("has_valid_location") == False).count()
invalid_rating = validation_df.filter(F.col("has_valid_rating") == False).count()

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"Total Records: {total_count}")
print(f"Missing/Invalid Coordinates: {missing_coords} ({missing_coords/total_count*100:.2f}%)")
print(f"Invalid/Missing Ratings: {invalid_rating} ({invalid_rating/total_count*100:.2f}%)")
print("=" * 60)

# Charging speed distribution
print("\nCharging Speed Distribution:")
validation_df.groupBy("charging_speed").count().orderBy("charging_speed").show()

# Show sample of flagged records
print("\nSample of records with data quality issues:")
validation_df.filter(
    (F.col("has_valid_location") == False) | (F.col("has_valid_rating") == False)
).select("station_name", "charger_rating", "charger_rating_kw", "latitude", "longitude", "has_valid_location", "has_valid_rating").show(10, truncate=False)